# Render interactive plots for the web.

This notebook can generate concise interactive html that can be rendered on the web for visualisation.

In [ ]:
# Import Packages
import os
import sys
import numpy as np
import pandas as pd
from bokeh.io import output_notebook
output_notebook()

import sys
sys.path.append(os.path.dirname(os.getcwd()))
from trappytv import TrappyTV
from trappytv import CellView

## Load one cell and add filters

In [ ]:
from analysis.outliers import drop_edge_nans_and_interpolate
from analysis.sg import apply_sg_bf_on_xy, hodrick_prescott_on_xy

datapaths_one_track = ["../data/2025_12_30/M4_2025_12_30.hd5"]

one_cell = CellView(datapaths_one_track[0], compute_speed=True)
one_cell.add_columns(inputs=["x_unrefined", "y_unrefined"], func=drop_edge_nans_and_interpolate, outputs=["xf", "yf"], max_length=5)
one_cell.add_columns(inputs=["xf", "yf"], func=apply_sg_bf_on_xy, outputs=["xf_11Hz", "yf_11Hz"], inplace=False, window_length=3, polyorder=2)
one_cell.add_columns(inputs=["xf", "yf"], func=apply_sg_bf_on_xy, outputs=["xf_3Hz", "yf_3Hz"], inplace=False, window_length=7, polyorder=3)
one_cell.add_columns(inputs=["xf_3Hz", "yf_3Hz"], func=hodrick_prescott_on_xy, outputs=["xf_long", "yf_long"], inplace=False, lambda_smooth=1e4, 
                 interpolate_if_necessary=True, include_cycles=False)
one_cell().columns

## Drop columns to reduce load

In [ ]:
one_cell.keep_columns(['frame', 'split', 'dt', 'gframe',
       'signal', 'ep', 'x', 'y', 'x_unrefined', 'y_unrefined',
       'particle','temp', 'speed', 'xf', 'yf',
       'postprocess', 'xf_11Hz', 'yf_11Hz', 'xf_3Hz', 'yf_3Hz', 'xf_long',
       'yf_long'])
one_cell().columns

one_cell.dfs["tracks"] = one_cell.dfs["tracks"][one_cell.dfs["tracks"].split < 10]

In [ ]:
plot_folder = os.path.abspath("../docs/interactive_plots/")
full_view_path = os.path.join(plot_folder, "full_view.html")
split_view_path = os.path.join(plot_folder, "split_view.html")
ensemble_view_path = os.path.join(plot_folder, "ensemble_view.html")

In [ ]:
#%%time
import bokeh
tv = TrappyTV(one_cell, width=1000, height=1000, filtered_columns={"filtered": ["xf", "yf"],
                                                                  "denoised": ["xf_11Hz", "yf_11Hz"],
                                                                  "anti-aliased": ["xf_3Hz", "yf_3Hz"],
                                                                  "long-term": ["xf_long", "yf_long"]})

## Render split view
tv.view_split(split_no=0)
bokeh.io.save(tv.layout, split_view_path)

## Render full view
tv.view_all(sample=10)
bokeh.io.save(tv.layout, full_view_path)

clear()

In [ ]:
## Delete tv and cell object
del tv
del one_cell

# Load Ensemble set

In [ ]:
from analysis.outliers import drop_edge_nans_and_interpolate
from analysis.sg import apply_sg_bf_on_xy, hodrick_prescott_on_xy

datapaths_many_track = ["../data/incubation_times_analysis_data/02122025.hd5"]

many_cells = CellView(datapaths_many_track[0], xycols=["x", "y"], compute_speed=False)
many_cells().columns

In [ ]:
## Remove open traps explicitly
many_cells.dfs["tracks"] = many_cells.dfs["tracks"][many_cells.dfs["tracks"].trap_open == False]
many_cells.dfs["tracks"] = many_cells.dfs["tracks"][many_cells.dfs["tracks"].split < 10]


## Trim un-necessary columns
many_cells.keep_columns(['x', 'y', 'signal','particle', 'split',
       'frame','temp', 'scopeid', 'eid', 'colony', 'group'])
many_cells().columns




In [ ]:
#%%time
## Slider does not work
tv = TrappyTV(many_cells, width=1000, height=1000, default_xycols=["x", "y"], hover_columns=[("x", "y"), "dt", "frame", "scopeid", "eid", "colony", "particle", "group"])
tv.view_ensemble(smooth_window=100)
bokeh.io.save(tv.layout, ensemble_view_path)


In [ ]:
del tv
del many_cells